# Task 4 CNN Workflow Notebook

This notebook uses `CNN.py` as context, but it is reorganized into separate stages so each part can be ran independently.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)


## Configuration And Paths

In [ ]:
current_dir = Path.cwd().resolve()
if (current_dir / "CNN.py").exists():
    TASK_DIR = current_dir
else:
    TASK_DIR = (current_dir / "Learning_Path" / "Task_4").resolve()

if not TASK_DIR.exists():
    raise RuntimeError("Could not locate Learning_Path/Task_4 from the current notebook working directory.")

BASE_DIR = TASK_DIR.parent
SAMPLES_DIR = BASE_DIR / "samples"
LABELS_DIR = BASE_DIR / "labels"
OUTPUT_DIR = TASK_DIR / "outputs"

IMAGE_SIZE = 64
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.001
PATIENCE = 10
SEED = 42

LOW_THRESHOLD = 150
MEDIUM_THRESHOLD = 180
CLASS_NAMES = ["low", "medium", "high"]

def set_random_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Task directory: {TASK_DIR}")
print(f"Running on: {device}")


## Data Helper Functions


In [ ]:
def label_to_class(label_tile):
    mean_ndvi = float(np.nanmean(label_tile))

    if mean_ndvi < LOW_THRESHOLD:
        return 0
    if mean_ndvi < MEDIUM_THRESHOLD:
        return 1
    return 2

def load_dataset(samples_dir, labels_dir, image_size):
    images = []
    labels = []

    for sample_path in sorted(samples_dir.glob("*.tif*")):
        label_path = labels_dir / sample_path.name.replace("img_", "ndvi_")

        if not label_path.exists():
            continue

        with rasterio.open(sample_path) as src:
            image = src.read(
                out_shape=(src.count, image_size, image_size),
                resampling=Resampling.bilinear,
            ).astype(np.float32)

        with rasterio.open(label_path) as src:
            label = src.read(1).astype(np.float32)

        image = image / 255.0
        image = np.nan_to_num(image)

        images.append(image)
        labels.append(label_to_class(label))

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)

    if len(images) == 0:
        raise RuntimeError("No image/label pairs were loaded.")

    return images, labels

def print_dataset_summary(images, labels):
    print(f"Total images: {len(images)}")
    print(f"Class counts: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=3)))}")

def split_dataset(images, labels, seed):
    train_images, temp_images, train_labels, temp_labels = train_test_split(
        images,
        labels,
        test_size=0.30,
        stratify=labels,
        random_state=seed,
    )

    val_images, test_images, val_labels, test_labels = train_test_split(
        temp_images,
        temp_labels,
        test_size=0.50,
        stratify=temp_labels,
        random_state=seed,
    )

    print(f"Train: {len(train_images)}")
    print(f"Validation: {len(val_images)}")
    print(f"Test: {len(test_images)}")

    return (
        train_images,
        val_images,
        test_images,
        train_labels,
        val_labels,
        test_labels,
    )

def compute_normalization_stats(train_images):
    mean = train_images.mean(axis=(0, 2, 3), keepdims=True)
    std = train_images.std(axis=(0, 2, 3), keepdims=True)
    std[std == 0] = 1.0
    return mean, std

def normalize_images(images, mean, std):
    return (images - mean) / std


## Load The Dataset


In [ ]:
set_random_seed(SEED)
images, labels = load_dataset(SAMPLES_DIR, LABELS_DIR, IMAGE_SIZE)
print_dataset_summary(images, labels)


## Split And Normalize


In [ ]:
(
    train_images,
    val_images,
    test_images,
    train_labels,
    val_labels,
    test_labels,
) = split_dataset(images, labels, SEED)

mean, std = compute_normalization_stats(train_images)
train_images = normalize_images(train_images, mean, std)
val_images = normalize_images(val_images, mean, std)
test_images = normalize_images(test_images, mean, std)


## Training Augmentation

These transforms are applied only to the training split, following the Task 4 notebook.


In [ ]:
# train_transforms = transforms.Compose([
#     transforms.RandomRotation(5),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomVerticalFlip(),
#     transforms.ColorJitter(brightness=0.05, contrast=0.05),
#     transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
# ])


## Dataset And DataLoaders


In [ ]:
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

def create_dataloaders(
    train_images,
    val_images,
    test_images,
    train_labels,
    val_labels,
    test_labels,
    batch_size,
    train_transform,
):
    train_data = AugmentedDataset(train_images, train_labels, transform=train_transform)
    val_data = AugmentedDataset(val_images, val_labels, transform=None)
    test_data = AugmentedDataset(test_images, test_labels, transform=None)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_data, val_data, test_data, train_loader, val_loader, test_loader


In [ ]:
train_data, val_data, test_data, train_loader, val_loader, test_loader = create_dataloaders(
    train_images,
    val_images,
    test_images,
    train_labels,
    val_labels,
    test_labels,
    BATCH_SIZE,
    train_transforms,
)


## Verify Augmentation

This is optional, but it helps confirm the training augmentation is actually changing a sample.


In [ ]:
num_trials = 10
non_zero_orig_aug = 0
non_zero_aug_aug = 0

for i in range(num_trials):
    orig = torch.tensor(train_images[0], dtype=torch.float32)
    aug1, _ = train_data[0]
    aug2, _ = train_data[0]

    diff_orig_aug1 = (orig - aug1).abs().mean().item()
    diff_aug1_aug2 = (aug1 - aug2).abs().mean().item()

    if diff_orig_aug1 > 0:
        non_zero_orig_aug += 1
    if diff_aug1_aug2 > 0:
        non_zero_aug_aug += 1

    print(f"Trial {i + 1:02d}: orig->aug1={diff_orig_aug1:.6f}, aug1->aug2={diff_aug1_aug2:.6f}")

print("\nSummary")
print(f"Non-zero orig->aug1 in {non_zero_orig_aug}/{num_trials} trials")
print(f"Non-zero aug1->aug2 in {non_zero_aug_aug}/{num_trials} trials")

if non_zero_orig_aug == 0 and non_zero_aug_aug == 0:
    print("Augmentation appears inactive for this sample.")
else:
    print("Augmentation is active.")


## Model Definition


In [ ]:
class VegetationCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


## Training Helpers


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, runtime_device):
    model.train()
    running_loss = 0.0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(runtime_device)
        batch_labels = batch_labels.to(runtime_device)

        optimizer.zero_grad()
        outputs = model(batch_images)
        loss = loss_fn(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

def validate_model(model, loader, loss_fn, runtime_device):
    model.eval()
    val_loss_total = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(runtime_device)
            batch_labels = batch_labels.to(runtime_device)

            outputs = model(batch_images)
            loss = loss_fn(outputs, batch_labels)
            val_loss_total += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

    val_loss = val_loss_total / len(loader)
    val_accuracy = correct / total
    return val_loss, val_accuracy

def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    runtime_device,
    epochs,
    patience,
    model_path,
):
    best_val_loss = float("inf")
    wait = 0

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, runtime_device)
        val_loss, val_accuracy = validate_model(model, val_loader, loss_fn, runtime_device)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            wait = 0
            torch.save(model.state_dict(), model_path)
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break


## Train The Model


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = OUTPUT_DIR / "basic_cnn.pth"

model = VegetationCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    device,
    EPOCHS,
    PATIENCE,
    best_model_path,
)


## Evaluation Helpers


In [ ]:
def evaluate_model(model, loader, runtime_device):
    model.eval()
    all_true = []
    all_pred = []

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(runtime_device)

            outputs = model(batch_images)
            _, predicted = torch.max(outputs, 1)

            all_true.extend(batch_labels.numpy())
            all_pred.extend(predicted.cpu().numpy())

    return {
        "accuracy": accuracy_score(all_true, all_pred),
        "precision": precision_score(all_true, all_pred, average="macro", zero_division=0),
        "recall": recall_score(all_true, all_pred, average="macro", zero_division=0),
        "f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
    }

def print_test_results(metrics):
    print("\nTest Results")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-Score : {metrics['f1']:.4f}")


## Evaluate The Best Checkpoint


In [ ]:
best_model = VegetationCNN().to(device)
best_model.load_state_dict(
    torch.load(best_model_path, map_location=device, weights_only=True)
)

metrics = evaluate_model(best_model, test_loader, device)
print_test_results(metrics)

print(f"\nSaved model: {best_model_path}")
